# Day 26: Customer-Feedback Intelligence Pipeline
Run in order with numpy, pandas, scikit-learn, sentence-transformers, transformers, and PyTorch installed. Model weights must be cached for offline use; otherwise the first download requires network access. Results depend on model versions and the small test set.


## Dataset and setup


In [ ]:
import warnings, re, textwrap
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd

RNG = 42
np.random.seed(RNG)

# 40 short customer reviews for a fictional gadget store "Nimbus".
# Each is (review_text, sentiment) where sentiment is the ground-truth label.
REVIEWS = [
    ("The delivery arrived two days early and the packaging was flawless.", "positive"),
    ("Battery life is incredible, easily lasts me a full working day.", "positive"),
    ("Setup took five minutes and the app walked me through every step.", "positive"),
    ("Customer support answered in under a minute and fixed my issue.", "positive"),
    ("Build quality feels premium, the aluminium body is gorgeous.", "positive"),
    ("Best purchase this year, the sound quality genuinely surprised me.", "positive"),
    ("Screen is bright and crisp even in direct sunlight, love it.", "positive"),
    ("Refund was processed instantly, no questions asked. Great service.", "positive"),
    ("The subscription is worth every penny, new features every month.", "positive"),
    ("Lightweight and comfortable, I forget I am even wearing it.", "positive"),
    ("Pairing with my laptop was seamless and it just works.", "positive"),
    ("Fantastic value for the price, would recommend to friends.", "positive"),
    ("The firmware update fixed the lag and now it is buttery smooth.", "positive"),
    ("Packaging was recyclable and the quick-start guide was clear.", "positive"),
    ("Warranty claim was handled in a single email, very impressed.", "positive"),
    ("Camera photos are sharp and colours look natural, thrilled.", "positive"),
    ("Delivery was five days late and no one told me why.", "negative"),
    ("The battery drains overnight even when the device is switched off.", "negative"),
    ("App keeps crashing every time I open the settings menu.", "negative"),
    ("Support put me on hold for forty minutes then dropped the call.", "negative"),
    ("The plastic casing cracked within a week of normal use.", "negative"),
    ("Overpriced for what it is, I expected far better at this cost.", "negative"),
    ("Screen has a dead pixel right in the middle, very distracting.", "negative"),
    ("They refused my refund and blamed me for the defect.", "negative"),
    ("The subscription auto-renewed without any warning email.", "negative"),
    ("It is heavy and uncomfortable after an hour of wearing it.", "negative"),
    ("Bluetooth disconnects randomly every few minutes, infuriating.", "negative"),
    ("Terrible value, it broke twice and shipping cost me extra.", "negative"),
    ("The latest firmware update bricked my device completely.", "negative"),
    ("Packaging was crushed and the manual was missing entirely.", "negative"),
    ("Warranty was denied because the form was 'filled in wrong'.", "negative"),
    ("Camera is grainy in low light and the autofocus hunts forever.", "negative"),
    ("Shipping was quick but the box arrived dented on one corner.", "negative"),
    ("Good sound but the app notifications are relentless and annoying.", "negative"),
    ("Love the design though the battery could honestly be better.", "positive"),
    ("Works well overall, only wish support replied a little faster.", "positive"),
    ("Cheap feeling buttons let down an otherwise solid product.", "negative"),
    ("The return shipping label never arrived so I am stuck with it.", "negative"),
    ("Crystal clear calls and the mic cancels background noise well.", "positive"),
    ("Constant software bugs make an expensive product feel like a beta.", "negative"),
]

df = pd.DataFrame(REVIEWS, columns=["text", "sentiment"])
print(df.shape)
print(df["sentiment"].value_counts())
df.head(3)

## Preprocessing


In [ ]:
STOP = {
    "the","a","an","and","or","but","is","are","was","were","it","its","i",
    "me","my","of","to","in","for","on","at","this","that","with","as","so",
    "even","then","far","than","after","every","no","not","up","out","you",
}

def normalize(text):
    text = text.lower()
    text = re.sub(r"[^a-z0-9\s]", " ", text)          # drop punctuation
    tokens = [t for t in text.split() if t and t not in STOP]
    return tokens

df["tokens"] = df["text"].apply(normalize)
df["clean"] = df["tokens"].apply(lambda ts: " ".join(ts))

print("RAW :", df.loc[0, "text"])
print("TOKS:", df.loc[0, "tokens"])
print("avg tokens/review:", round(df["tokens"].str.len().mean(), 1))

## TF-IDF sentiment baseline


In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report

y = (df["sentiment"] == "positive").astype(int)   # 1 = positive, 0 = negative
Xtr, Xte, ytr, yte, idx_tr, idx_te = train_test_split(
    df["clean"], y, df.index, test_size=0.30, stratify=y, random_state=RNG)

vec = TfidfVectorizer(ngram_range=(1, 2), min_df=1)
Xtr_v = vec.fit_transform(Xtr)
Xte_v = vec.transform(Xte)

clf = LogisticRegression(C=1.0, max_iter=2000).fit(Xtr_v, ytr)
pred = clf.predict(Xte_v)
print(f"test accuracy = {accuracy_score(yte, pred):.3f}  (n_test={len(yte)})")

# Which words push the decision toward positive vs negative?
vocab = np.array(vec.get_feature_names_out())
coefs = clf.coef_[0]
top_pos = vocab[np.argsort(coefs)[-6:][::-1]]
top_neg = vocab[np.argsort(coefs)[:6]]
print("most POSITIVE features:", list(top_pos))
print("most NEGATIVE features:", list(top_neg))

## Semantic search


In [ ]:
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

embedder = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2", device="cpu")
emb = embedder.encode(df["text"].tolist(), normalize_embeddings=True,
                      show_progress_bar=False)
print("embedding matrix:", emb.shape)

def search(query, k=3):
    q = embedder.encode([query], normalize_embeddings=True)
    sims = cosine_similarity(q, emb)[0]
    order = np.argsort(sims)[::-1][:k]
    return [(round(float(sims[i]), 3), df.loc[i, "text"]) for i in order]

print("\nQuery: 'problems with the battery'")
for score, txt in search("problems with the battery"):
    print(f"  {score}  {txt}")

print("\nQuery: 'the shipping was slow'")
for score, txt in search("the shipping was slow"):
    print(f"  {score}  {txt}")

## Complaint summary


In [ ]:
from transformers import pipeline

summarizer = pipeline("text2text-generation",
                      model="google/flan-t5-small", device=-1)  # -1 = CPU

neg_text = " ".join(df.loc[df["sentiment"] == "negative", "text"].tolist())
prompt = ("Summarize the main problems customers complain about in these "
          "product reviews, in one short paragraph:\n" + neg_text)

out = summarizer(prompt, max_new_tokens=80, do_sample=False)[0]["generated_text"]
print("SUMMARY OF COMPLAINTS:\n", textwrap.fill(out, 88))

## Topic labeling


In [ ]:
TOPICS = ["shipping", "battery", "app/software", "support", "build quality", "price"]

def label_topic(review):
    prompt = (
        "Classify the customer review into exactly one topic from this list: "
        + ", ".join(TOPICS) + ".\n"
        "Review: \"" + review + "\"\n"
        "Topic:"
    )
    ans = summarizer(prompt, max_new_tokens=8, do_sample=False)[0]["generated_text"]
    ans = ans.strip().lower()
    # snap the free-text answer back onto the allowed set
    for t in TOPICS:
        if t.split("/")[0] in ans:
            return t
    return "other"

df["topic"] = df["text"].apply(label_topic)
print(df[["text", "sentiment", "topic"]].head(8).to_string(index=False))
print("\nTopic distribution:")
print(df["topic"].value_counts())

## Dashboard


In [ ]:
n = len(df)
pos = int((df["sentiment"] == "positive").sum())
neg = n - pos

print("="*60)
print("  NIMBUS — CUSTOMER FEEDBACK DASHBOARD (offline pipeline)")
print("="*60)
print(f"Reviews analysed      : {n}")
print(f"Sentiment split       : {pos} positive ({pos/n:.0%})  |  "
      f"{neg} negative ({neg/n:.0%})")

neg_by_topic = (df[df["sentiment"] == "negative"]
                .groupby("topic").size().sort_values(ascending=False))
print("\nTop complaint areas (negative reviews by topic):")
for topic, cnt in neg_by_topic.items():
    bar = "#" * int(cnt)
    print(f"  {topic:14s} {cnt:2d}  {bar}")

if len(neg_by_topic):
    worst = neg_by_topic.index[0]
    print(f"\n>> Loudest problem: '{worst}'. "
          f"Example: {df[(df.sentiment=='negative') & (df.topic==worst)].iloc[0]['text']!r}")

## Comparison


In [ ]:
print("SAME REVIEW, THREE LENSES\n" + "-"*60)
sample = "Love the design though the battery could honestly be better."
print("Review:", sample, "\n")

# 1) Classical: TF-IDF logistic-regression sentiment
p = clf.predict_proba(vec.transform([" ".join(normalize(sample))]))[0][1]
print(f"1) CLASSICAL (TF-IDF+LogReg)  -> P(positive) = {p:.2f}")

# 2) Embedding: nearest neighbour by meaning
print("2) EMBEDDING (MiniLM search) -> closest other review:")
for score, txt in search(sample, k=2)[1:]:
    print(f"     {score}  {txt}")

# 3) LLM: zero-shot topic + generative reasoning
print(f"3) LLM (flan-t5 zero-shot)   -> topic = {label_topic(sample)}")

print("\n" + "-"*60)
print(textwrap.dedent("""\
    ANSWER THESE IN A NEW MARKDOWN CELL:
    A. Where was TF-IDF *enough*, and where did it clearly fall short?
    B. What did embeddings let you do that the bag-of-words baseline could not?
    C. What did the LLM add that neither classical NLP nor raw embeddings gave you?
    D. This review is mixed ('love the design' + 'battery could be better').
       Which of the three families handled that nuance best, and why?
    E. Cost/latency: rank the three by compute needed. When would you still
       ship the *classical* model to production instead of the LLM?
"""))

## Reflection
TF-IDF with logistic regression provides an inexpensive, interpretable baseline, but sparse word overlap makes paraphrases and mixed sentiment difficult. Embeddings support semantic retrieval, although similarities do not explain causality. An instruction-tuned model adds natural-language summaries and topic labels without topic training examples; its outputs still need checking against source reviews. Mixed reviews contain multiple aspects, so a single sentiment or topic label necessarily loses information. The baseline is a sensible production choice when latency, cost, interpretability, and measured held-out performance meet requirements. The dataset is tiny and the dashboard uses supplied sentiment labels, not predictions. Review actual outputs before drawing conclusions.
